# PCA on Gene Expression Data

## BEE-102 Assignment

Reproducing Figure 1 from the [Nature Primer on PCA](https://www.nature.com/articles/nbt0308-303).

Gene Expression Omnibus (GEO) ID: [GSE5325](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE5325) — 105 patient samples (ER+ and ER- breast cancer).

In [ ]:
# --- CELL 1: IMPORTS ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

In [ ]:
# --- CELL 2: DATA LOADING & PREPROCESSING ---

# 1. Load class labels (1 = ER+, 0 = ER-)
labels_df = pd.read_csv('data/class.tsv', header=None, names=['ER_Status'])
y = labels_df['ER_Status'].values
print(f"Labels loaded: {len(y)} samples, {sum(y==1)} ER+, {sum(y==0)} ER-")

# 2. Load gene mapping (has comment lines starting with '#')
columns_df = pd.read_csv('data/columns.tsv.gz', sep='\t', comment='#')
print(f"Gene mapping columns: {columns_df.columns.tolist()}")
print(f"Total gene probes: {len(columns_df)}")

# 3. Load the full expression data (105 patients x 16174 genes)
expr_df = pd.read_csv('data/filtered.tsv.gz', sep='\t')
expr_df.columns = expr_df.columns.astype(str).str.strip()
X_full = expr_df.values
print(f"Expression matrix shape: {X_full.shape}")

# 4. Find specific gene IDs
# GATA3 -> ID 4359, XBP1 -> ID 4404, CCNB2 -> ID 4459
gata3_row = columns_df.loc[columns_df['GeneSymbol'] == 'GATA3'].iloc[0]
xbp1_row = columns_df.loc[columns_df['GeneSymbol'] == 'XBP1'].iloc[0]
ccnb2_row = columns_df.loc[columns_df['GeneSymbol'] == 'CCNB2'].iloc[0]

gata3_id = str(gata3_row['ID'])
xbp1_id = str(xbp1_row['ID'])
ccnb2_id = str(ccnb2_row['ID'])

print(f"GATA3 ID: {gata3_id}, XBP1 ID: {xbp1_id}, CCNB2 ID: {ccnb2_id}")

# 5. Extract 2D dataset: GATA3 (x-axis) and XBP1 (y-axis)
gata3_vals = expr_df[gata3_id].values
xbp1_vals = expr_df[xbp1_id].values
X_2d = np.column_stack([gata3_vals, xbp1_vals])

# Get column indices for later use in biplot
all_cols = list(expr_df.columns)
xbp1_col_idx = all_cols.index(xbp1_id)
ccnb2_col_idx = all_cols.index(ccnb2_id)

# Boolean masks for ER status
er_pos = (y == 1)
er_neg = (y == 0)

print(f"2D dataset shape: {X_2d.shape}")
print(f"XBP1 column index: {xbp1_col_idx}, CCNB2 column index: {ccnb2_col_idx}")

In [ ]:
# --- CELL 3: MANUAL PCA IMPLEMENTATION (using SVD) ---

def manual_pca(X, n_components=None):
    """
    Performs Principal Component Analysis using NumPy SVD.
    Returns: scores, components (Vt), singular_values, explained_variance,
             explained_variance_ratio, mean_vector
    """
    n_samples = X.shape[0]
    
    # Step 1: Mean-center the data
    mean_vec = np.mean(X, axis=0)
    X_centered = X - mean_vec
    
    # Step 2: SVD decomposition
    # X_centered = U * diag(S) * Vt
    U, S, Vt = np.linalg.svd(X_centered, full_matrices=False)
    
    # Step 3: Calculate scores (projected data)
    scores = U * S  # equivalent to X_centered @ Vt.T
    
    # Step 4: Explained variance
    explained_variance = (S ** 2) / (n_samples - 1)
    explained_variance_ratio = explained_variance / np.sum(explained_variance)
    
    if n_components is not None:
        scores = scores[:, :n_components]
        Vt = Vt[:n_components, :]
        S = S[:n_components]
        explained_variance = explained_variance[:n_components]
        explained_variance_ratio = explained_variance_ratio[:n_components]
    
    return scores, Vt, S, explained_variance, explained_variance_ratio, mean_vec

print("Manual PCA function defined.")

In [ ]:
# --- CELL 4: EXECUTE PCA + FIX SIGN DIRECTION ---

# PCA on 2D toy data (GATA3, XBP1) for panels a, b, c
scores_2d, Vt_2d, S_2d, expvar_2d, _, mean_2d = manual_pca(X_2d, n_components=2)

# FIX PC1 SIGN: In the reference figure, ER+ samples (high GATA3/XBP1) 
# project to the RIGHT (positive) side of PC1.
# If SVD gave us the opposite direction, flip it.
if np.mean(scores_2d[er_pos, 0]) < np.mean(scores_2d[er_neg, 0]):
    scores_2d[:, 0] *= -1
    Vt_2d[0, :] *= -1
    print("Flipped PC1 sign for 2D PCA (to match reference direction)")

# PCA on full dataset for panels d, e, f
scores_full, Vt_full, S_full, _, expvar_ratio_full, _ = manual_pca(X_full)

# FIX PC1 SIGN for full PCA too
if np.mean(scores_full[er_pos, 0]) < np.mean(scores_full[er_neg, 0]):
    scores_full[:, 0] *= -1
    Vt_full[0, :] *= -1
    print("Flipped PC1 sign for full PCA (to match reference direction)")

print(f"2D PCA scores shape: {scores_2d.shape}")
print(f"Full PCA scores shape: {scores_full.shape}")
print(f"Mean PC1 for ER+: {np.mean(scores_2d[er_pos, 0]):.3f}")
print(f"Mean PC1 for ER-: {np.mean(scores_2d[er_neg, 0]):.3f}")
print("ER+ is now on the RIGHT side of PC1 ✓")

In [ ]:
# --- CELL 5: GENERATE ALL 6 PANELS ---

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
plt.subplots_adjust(wspace=0.35, hspace=0.45)

c_pos = '#E41A1C'  # Red for ER+
c_neg = '#000000'  # Black for ER-
mk = 's'           # Square markers (same as reference)
ms = 18            # Marker size

def styled_scatter(ax, x, y_data, xlabel, ylabel, title):
    """Helper to make scatter plots matching reference style."""
    ax.scatter(x[er_neg], y_data[er_neg], c=c_neg, s=ms, marker=mk, zorder=2)
    ax.scatter(x[er_pos], y_data[er_pos], c=c_pos, s=ms, marker=mk, zorder=2)
    ax.set_xlabel(xlabel, fontsize=12)
    ax.set_ylabel(ylabel, fontsize=12)
    ax.set_title(title, loc='left', fontweight='bold', fontsize=16)
    ax.spines[['top', 'right']].set_visible(False)

# ============================================================
# Panel a: Original 2D scatter (GATA3 vs XBP1)
# ============================================================
ax = axes[0, 0]
styled_scatter(ax, X_2d[:, 0], X_2d[:, 1], 'GATA3', 'XBP1', 'a')

# ============================================================
# Panel b: 2D scatter with PCA direction vectors
# ============================================================
ax = axes[0, 1]
styled_scatter(ax, X_2d[:, 0], X_2d[:, 1], 'GATA3', 'XBP1', 'b')

# Draw PC1 and PC2 arrows from the data mean
mx, my = mean_2d
for i in range(2):
    vec = Vt_2d[i] * np.sqrt(expvar_2d[i]) * 2.5
    label = f'PC{i+1}'
    ax.annotate(label, xy=(mx + vec[0], my + vec[1]), xytext=(mx, my),
                arrowprops=dict(arrowstyle='->', lw=2, color='black'),
                fontsize=12, fontweight='bold')

# ============================================================
# Panel c: Projection onto PC1 (strip chart)
# ============================================================
ax = axes[0, 2]
pc1 = scores_2d[:, 0]

# Row 1 (top, y=3): All samples
ax.scatter(pc1[er_neg], np.full(sum(er_neg), 3), c=c_neg, s=ms, marker=mk)
ax.scatter(pc1[er_pos], np.full(sum(er_pos), 3), c=c_pos, s=ms, marker=mk)

# Row 2 (middle, y=2): ER- only
ax.scatter(pc1[er_neg], np.full(sum(er_neg), 2), c=c_neg, s=ms, marker=mk)

# Row 3 (bottom, y=1): ER+ only
ax.scatter(pc1[er_pos], np.full(sum(er_pos), 1), c=c_pos, s=ms, marker=mk)

ax.set_yticks([1, 2, 3])
ax.set_yticklabels(['ER$^+$', 'ER$^-$', 'All'], fontsize=11)
ax.set_xlabel('Projection onto PC1', fontsize=12)
ax.set_title('c', loc='left', fontweight='bold', fontsize=16)
ax.spines[['top', 'right', 'left']].set_visible(False)
ax.set_ylim(0.5, 3.5)
for yy in [1, 2, 3]:
    ax.axhline(yy, color='lightgray', lw=0.8, zorder=0)

# ============================================================
# Panel d: Scree Plot (full PCA)
# ============================================================
ax = axes[1, 0]
var_pct = expvar_ratio_full * 100
ax.bar(range(1, len(var_pct) + 1), var_pct, color='gray', edgecolor='black', linewidth=0.3)
ax.set_xlabel('Principal component', fontsize=12)
ax.set_ylabel('Proportion of variance (%)', fontsize=12)
ax.set_xlim(0, len(var_pct) + 1)
ax.set_title('d', loc='left', fontweight='bold', fontsize=16)
ax.spines[['top', 'right']].set_visible(False)

# ============================================================
# Panel e: Biplot (PCA scores + gene loadings)
# ============================================================
ax = axes[1, 1]
styled_scatter(ax, scores_full[:, 0], scores_full[:, 1],
               'Projection onto PC1', 'Projection onto PC2', 'e')

# Gene loading vectors (scale for visibility)
scale = 250
xbp1_load = Vt_full[:2, xbp1_col_idx] * scale
ccnb2_load = Vt_full[:2, ccnb2_col_idx] * scale

ax.plot([0, xbp1_load[0]], [0, xbp1_load[1]], 'k-', lw=1)
ax.plot([0, ccnb2_load[0]], [0, ccnb2_load[1]], 'k-', lw=1)
ax.scatter([xbp1_load[0]], [xbp1_load[1]], c='#2CA02C', s=60, zorder=5)
ax.scatter([ccnb2_load[0]], [ccnb2_load[1]], c='#2CA02C', s=60, zorder=5)
ax.text(xbp1_load[0] + 2, xbp1_load[1] + 2, 'XBP1', fontsize=11, fontstyle='italic')
ax.text(ccnb2_load[0] + 2, ccnb2_load[1] - 4, 'CCNB2', fontsize=11, fontstyle='italic')

# ============================================================
# Panel f: Multi-class view (KMeans clusters as proxy)
# ============================================================
ax = axes[1, 2]
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10).fit(scores_full[:, :5])
cluster_labels = kmeans.labels_

cluster_colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
for cid in range(4):
    mask = (cluster_labels == cid)
    ax.scatter(scores_full[mask, 0], scores_full[mask, 1],
              c=cluster_colors[cid], s=ms, marker=mk, label=f'Cluster {cid+1}')

ax.set_xlabel('Projection onto PC1', fontsize=12)
ax.set_ylabel('Projection onto PC2', fontsize=12)
ax.set_title('f', loc='left', fontweight='bold', fontsize=16)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('pca_figure1.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved as pca_figure1.png")